# Byzantine Stress Test: Algorithmic Breakdown Point Analysis
### Investigating Robust Aggregation Failure & Agentic Resilience ($f=3$ and $f=4$ Attackers)

This notebook evaluates the **breakdown point** of conventional robust aggregation (`FixedTrimmedMean`) versus our proposed `AgenticAI` under intensified Byzantine adversarial infiltration ($f=3$ and $f=4$ attackers on $N=10$ clients).

### Scientific Motivation & Hypothesis:
1. **Trimmed Mean's Breakdown Point is Exceeded**: With $N=10$ clients and trim ratio $q=0.2$, Trimmed Mean mechanically trims $k = \lfloor 10 \times 0.2 \rfloor = 2$ updates from each coordinate.
   - At **$f=2$ (20% infiltration)**: Both attackers are eliminated $\to$ Trimmed Mean survives (91.82% in previous experiment).
   - At **$f=3$ (30% infiltration)**: Exactly 1 poisoned update leaks into the trimmed mean coordinate $\to$ **catastrophic collapse**.
   - At **$f=4$ (40% infiltration)**: 2 poisoned updates leak into the trimmed mean coordinate $\to$ **catastrophic collapse**.
2. **FixedFedAvg**: Zero Byzantine resilience $\to$ **immediate collapse**.
3. **AgenticAI**: Dynamically flags all 3 or 4 anomalous client profiles, evaluates candidate clean subsets, and selects clean-subset aggregation $\to$ **maintains high accuracy and survives**.

### Quick Setup Instructions:
1. **GPU**: Select `Runtime` $\to$ `Change runtime type` $\to$ `T4 GPU`.
2. **API Key**: Click the Key icon (Secrets) in the left sidebar $\to$ add `GROQ_API_KEY`.
3. Click **Runtime** $\to$ **Run all**.

## 0. Mount Google Drive (Auto-Backup)

In [10]:
from google.colab import drive
import os

drive.mount('/content/drive')
print("Google Drive mounted successfully at /content/drive.")
print("Stress test databases will be backed up to: /content/drive/MyDrive/")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully at /content/drive.
Stress test databases will be backed up to: /content/drive/MyDrive/


In [27]:
import sqlite3, os, time, datetime
from IPython.display import clear_output

db_f4 = '/content/fl_project/fl_metrics_byzantine_stress_f4.db'
all_methods = ['FixedFedAvg', 'FixedTrimmedMean', 'AgenticAI']

print("Starting live monitor (press the Stop button anytime to exit)...")

try:
    while True:
        clear_output(wait=True)
        now = datetime.datetime.now().strftime('%H:%M:%S')

        print("=" * 70)
        print(f" LIVE EXPERIMENT MONITOR (f=4 Attackers) — Last updated: {now}")
        print("=" * 70)

        if not os.path.exists(db_f4):
            print("⏳ Database initializing... Waiting for first round to record.")
            time.sleep(5)
            continue

        con = sqlite3.connect(db_f4)
        rows = con.execute("""
            SELECT runs.method,
                   COUNT(r.id) as rounds_done,
                   MAX(r.round) as latest_round,
                   ROUND(r.test_accuracy, 2) as current_acc,
                   ROUND(MAX(r.test_accuracy), 2) as max_acc
            FROM experiment3_rounds r
            JOIN experiment3_runs runs ON r.run_id = runs.run_id
            GROUP BY runs.method
        """).fetchall()
        con.close()

        data = {r[0]: (r[1], r[2], r[3], r[4]) for r in rows}
        total_done = sum(d[0] for d in data.values())

        print(f"{'Method':<20} | {'Progress':<12} | {'Latest Rnd':<10} | {'Current Acc':<12} | {'Max Acc'}")
        print("-" * 70)

        for m in all_methods:
            if m in data:
                done, latest, curr_acc, max_acc = data[m]
                pct = int((done / 20) * 10)
                bar = f"[{'=' * pct}{' ' * (10 - pct)}]"
                status = "DONE ✅" if done == 20 else f"{bar} {done}/20"
                print(f"{m:<20} | {status:<12} | R{latest:<9} | {curr_acc:>9.2f}% | {max_acc:>6.2f}%")
            else:
                print(f"{m:<20} | {'[queued]':<12} | {'--':<10} | {'--':<12} | {'--'}")

        print("-" * 70)
        print(f"Total Progress: {total_done} / 60 rounds completed ({total_done/60*100:.1f}%)")

        if total_done == 60:
            print("\n🎉 ALL 60 ROUNDS COMPLETE! You can now proceed to Phase 3.")
            break

        time.sleep(10)  # Refresh every 10 seconds

except KeyboardInterrupt:
    print("\n[Stopped live monitor]")

 LIVE EXPERIMENT MONITOR (f=4 Attackers) — Last updated: 18:53:56
Method               | Progress     | Latest Rnd | Current Acc  | Max Acc
----------------------------------------------------------------------
FixedFedAvg          | [====      ] 8/20 | R8         |     90.66% |  90.66%
FixedTrimmedMean     | [queued]     | --         | --           | --
AgenticAI            | [queued]     | --         | --           | --
----------------------------------------------------------------------
Total Progress: 8 / 60 rounds completed (13.3%)

[Stopped live monitor]


## 1. Verify GPU

In [12]:
!nvidia-smi


Fri Sep 18 17:19:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [13]:
!pip install -q torch torchvision numpy pydantic groq python-dotenv tabulate matplotlib


## 3. Clone Repository or Extract Project

In [14]:
import os, sys, shutil, zipfile

# Priority 1: Check if fl_project_colab.zip was uploaded manually
zip_path = '/content/fl_project_colab.zip'
if os.path.exists(zip_path):
    print("Found uploaded fl_project_colab.zip. Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content')
    print("Extracted project files to /content/fl_project.")
else:
    # Priority 2: Clone fresh from GitHub
    print("No zip found. Cloning repository from GitHub...")
    if os.path.exists('/content/Agentic-AI'):
        shutil.rmtree('/content/Agentic-AI')
    !git clone https://github.com/the-protag0n1st/Agentic-AI.git /content/Agentic-AI

    src_dir = '/content/Agentic-AI/fl_project_final/fl_project'
    dst_dir = '/content/fl_project'
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    print("Project successfully cloned and prepared at /content/fl_project.")

sys.path.insert(0, '/content/fl_project')
os.chdir('/content/fl_project')
print(f"Working directory: {os.getcwd()}")


No zip found. Cloning repository from GitHub...
Cloning into '/content/Agentic-AI'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 131 (delta 33), reused 119 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 678.54 KiB | 8.93 MiB/s, done.
Resolving deltas: 100% (33/33), done.
Project successfully cloned and prepared at /content/fl_project.
Working directory: /content/fl_project


## 4. Configure Groq API Key & Environment

In [15]:
import os, torch
from google.colab import userdata

try:
    groq_key = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = groq_key
    print("[OK] GROQ_API_KEY successfully loaded from Colab Secrets.")
except Exception:
    if not os.environ.get('GROQ_API_KEY'):
        import getpass
        os.environ['GROQ_API_KEY'] = getpass.getpass("Enter your GROQ_API_KEY: ")

print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")


[OK] GROQ_API_KEY successfully loaded from Colab Secrets.
PyTorch Version: 2.11.0+cu128 | CUDA Available: True
Active GPU: Tesla T4


---
## Phase 1 — Stress Test with $f=3$ Attackers (30% Infiltration)
Attacking clients: 3 out of 10 clients inject $-3.0\times$ scaled opposite updates in Rounds 16–20.
- `FixedFedAvg`: Baseline collapse
- `FixedTrimmedMean`: Breakdown point exceeded (only 2 updates trimmed $\to$ 1 poison update leaks)
- `AgenticAI`: Dynamic detection & clean-subset selection

In [16]:
import subprocess, sys, shutil, datetime, os

db_f3 = '/content/fl_project/fl_metrics_byzantine_stress_f3.db'
drive_f3 = '/content/drive/MyDrive/fl_metrics_byzantine_stress_f3.db'

if os.path.exists(drive_f3) and not os.path.exists(db_f3):
    shutil.copy2(drive_f3, db_f3)
    print("Restored f=3 DB from Google Drive.")

def backup_f3():
    try:
        if os.path.exists(db_f3):
            shutil.copy2(db_f3, drive_f3)
            ts = datetime.datetime.now().strftime('%H:%M:%S')
            print(f"[{ts}] DB f=3 successfully backed up to Drive ({os.path.getsize(drive_f3):,} bytes)")
    except Exception as e:
        print(f"Drive backup notice: {e}")

print("=" * 70)
print("RUNNING BASELINES (f=3 Attackers): FixedFedAvg, FixedTrimmedMean")
print("=" * 70)
subprocess.run([
    sys.executable, '-u', 'run_experiment3.py',   # <-- '-u' added here for live streaming output!
    '--seed', '1',
    '--methods', 'FixedFedAvg', 'FixedTrimmedMean',
    '--num-attackers', '3',
    '--rounds', '20',
    '--resume',
    '--db', db_f3,
    '--data-root', '/content/fl_project/data'
], check=True)
backup_f3()

RUNNING BASELINES (f=3 Attackers): FixedFedAvg, FixedTrimmedMean
[18:12:55] DB f=3 successfully backed up to Drive (87,842,816 bytes)


In [20]:
print("=" * 70)
print("RUNNING AGENTIC AI (f=3 Attackers): Adaptive Pruning & Aggregation")
print("=" * 70)
subprocess.run([
    sys.executable, 'run_experiment3.py',
    '--seed', '1',
    '--methods', 'AgenticAI',
    '--num-attackers', '3',
    '--rounds', '20',
    '--resume',
    '--db', db_f3,
    '--data-root', '/content/fl_project/data'
], check=True)
backup_f3()
print("\n[PHASE 1 COMPLETE] f=3 runs finished!")


RUNNING AGENTIC AI (f=3 Attackers): Adaptive Pruning & Aggregation
[18:39:30] DB f=3 successfully backed up to Drive (132,153,344 bytes)

[PHASE 1 COMPLETE] f=3 runs finished!


In [23]:
import os, sqlite3, datetime
import pandas as pd
from google.colab import files

drive_path = '/content/drive/MyDrive/fl_metrics_byzantine_stress_f3.db'
local_path = '/content/fl_project/fl_metrics_byzantine_stress_f3.db'

# ==============================================================================
# 1. VERIFY GOOGLE DRIVE BACKUP FILE
# ==============================================================================
print("=" * 80)
print("1. VERIFYING GOOGLE DRIVE BACKUP")
print("=" * 80)
if os.path.exists(drive_path):
    sz = os.path.getsize(drive_path)
    mtime = datetime.datetime.fromtimestamp(os.path.getmtime(drive_path)).strftime('%Y-%m-%d %H:%M:%S')
    print(f"✅ CONFIRMED: File is safely in your Google Drive!")
    print(f"   Location: {drive_path}")
    print(f"   Size: {sz:,} bytes ({sz / (1024*1024):.1f} MB)")
    print(f"   Last Modified: {mtime}")
else:
    print(f"⚠️ Drive file not found. Copying from local Colab storage now...")
    import shutil
    shutil.copy2(local_path, drive_path)
    print(f"✅ Copied to Google Drive: {drive_path}")

active_db = drive_path if os.path.exists(drive_path) else local_path

# ==============================================================================
# 2. PRINT FULL ROUND-BY-ROUND RESULTS (ROUNDS 1 TO 20)
# ==============================================================================
print("\n" + "=" * 90)
print("2. COMPLETE ROUND-BY-ROUND RESULTS TABLE (f=3 Attackers, Seed 1)")
print("=" * 90)
con = sqlite3.connect(active_db)

df_rounds = pd.read_sql_query("""
    SELECT r.round, r.alpha, r.attack_active, runs.method, r.test_accuracy, r.aggregation_method, r.selected_clients
    FROM experiment3_rounds r
    JOIN experiment3_runs runs ON r.run_id = runs.run_id
    ORDER BY r.round ASC, runs.method ASC
""", con)

pivot_acc = df_rounds.pivot(index='round', columns='method', values='test_accuracy')
agentic_info = df_rounds[df_rounds['method'] == 'AgenticAI'].set_index('round')

print(f"{'Round':<6} | {'Regime':<12} | {'Attack':<7} | {'FixedFedAvg':<12} | {'TrimmedMean':<12} | {'AgenticAI (Ours)':<16} | Agentic Selection")
print("-" * 95)
for r in range(1, 21):
    regime = "Near-IID" if r <= 5 else ("Mod-NonIID" if r <= 10 else ("Sev-NonIID" if r <= 15 else "3-ATTACKERS"))
    atk = "YES" if r >= 16 else "no"
    acc_fed = f"{pivot_acc.loc[r, 'FixedFedAvg']:.2f}%" if 'FixedFedAvg' in pivot_acc.columns and r in pivot_acc.index else "N/A"
    acc_trm = f"{pivot_acc.loc[r, 'FixedTrimmedMean']:.2f}%" if 'FixedTrimmedMean' in pivot_acc.columns and r in pivot_acc.index else "N/A"
    acc_agy = f"{pivot_acc.loc[r, 'AgenticAI']:.2f}%" if 'AgenticAI' in pivot_acc.columns and r in pivot_acc.index else "N/A"

    sel_desc = ""
    if r in agentic_info.index:
        agg = agentic_info.loc[r, 'aggregation_method']
        clients = agentic_info.loc[r, 'selected_clients']
        sel_desc = f"{agg} on {clients}"

    print(f"R{r:<5} | {regime:<12} | {atk:<7} | {acc_fed:<12} | {acc_trm:<12} | {acc_agy:<16} | {sel_desc}")

# ==============================================================================
# 3. EXPORT COMPLETE DATA TO CSV FILES
# ==============================================================================
print("\n" + "=" * 80)
print("3. EXPORTING RAW TABLES TO CSV")
print("=" * 80)

df_all_rounds = pd.read_sql_query("SELECT * FROM experiment3_rounds", con)
df_all_rounds.to_csv("f3_rounds_complete.csv", index=False)
print(f"✅ Exported f3_rounds_complete.csv ({len(df_all_rounds)} rows)")

df_all_runs = pd.read_sql_query("SELECT * FROM experiment3_runs", con)
df_all_runs.to_csv("f3_runs_complete.csv", index=False)
print(f"✅ Exported f3_runs_complete.csv ({len(df_all_runs)} rows)")

try:
    df_agentic = pd.read_sql_query("SELECT * FROM agentic_log", con)
    df_agentic.to_csv("f3_agentic_decisions.csv", index=False)
    print(f"✅ Exported f3_agentic_decisions.csv ({len(df_agentic)} rows)")
except Exception as e:
    pass

try:
    df_llm = pd.read_sql_query("SELECT * FROM llm_call_log", con)
    df_llm.to_csv("f3_llm_calls.csv", index=False)
    print(f"✅ Exported f3_llm_calls.csv ({len(df_llm)} rows)")
except Exception as e:
    pass

con.close()

# ==============================================================================
# 4. DOWNLOAD DATABASE & CSVs TO YOUR LOCAL PC DIRECTLY
# ==============================================================================
print("\n" + "=" * 80)
print("4. DOWNLOADING FILES TO YOUR COMPUTER")
print("=" * 80)
files.download("f3_rounds_complete.csv")
files.download("f3_runs_complete.csv")
if os.path.exists("f3_agentic_decisions.csv"):
    files.download("f3_agentic_decisions.csv")
files.download(active_db)
print("✅ Browser download triggered for all CSVs and the .db file!")

1. VERIFYING GOOGLE DRIVE BACKUP
✅ CONFIRMED: File is safely in your Google Drive!
   Location: /content/drive/MyDrive/fl_metrics_byzantine_stress_f3.db
   Size: 132,153,344 bytes (126.0 MB)
   Last Modified: 2026-09-18 18:39:28

2. COMPLETE ROUND-BY-ROUND RESULTS TABLE (f=3 Attackers, Seed 1)
Round  | Regime       | Attack  | FixedFedAvg  | TrimmedMean  | AgenticAI (Ours) | Agentic Selection
-----------------------------------------------------------------------------------------------
R1     | Near-IID     | no      | 85.25%       | 84.77%       | 84.95%           | FedAvg on [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
R2     | Near-IID     | no      | 84.15%       | 85.41%       | 83.77%           | FedAvg on [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
R3     | Near-IID     | no      | 86.66%       | 87.86%       | 91.71%           | Median on [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
R4     | Near-IID     | no      | 87.54%       | 88.24%       | 91.70%           | Median on [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
R5     | Nea

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Browser download triggered for all CSVs and the .db file!


---
## Phase 2 — Stress Test with $f=4$ Attackers (40% Infiltration)
Attacking clients: 4 out of 10 clients inject $-3.0\times$ scaled opposite updates in Rounds 16–20.
- `FixedFedAvg`: Baseline collapse
- `FixedTrimmedMean`: Severe breakdown (only 2 updates trimmed $\to$ 2 poison updates leak)
- `AgenticAI`: Dynamic detection & clean-subset selection of remaining 6 clean clients

In [ ]:
db_f4 = '/content/fl_project/fl_metrics_byzantine_stress_f4.db'
drive_f4 = '/content/drive/MyDrive/fl_metrics_byzantine_stress_f4.db'

if os.path.exists(drive_f4) and not os.path.exists(db_f4):
    shutil.copy2(drive_f4, db_f4)
    print("Restored f=4 DB from Google Drive.")

def backup_f4():
    try:
        if os.path.exists(db_f4):
            shutil.copy2(db_f4, drive_f4)
            ts = datetime.datetime.now().strftime('%H:%M:%S')
            print(f"[{ts}] DB f=4 successfully backed up to Drive ({os.path.getsize(drive_f4):,} bytes)")
    except Exception as e:
        print(f"Drive backup notice: {e}")

print("=" * 70)
print("RUNNING BASELINES (f=4 Attackers): FixedFedAvg, FixedTrimmedMean")
print("=" * 70)
subprocess.run([
    sys.executable, 'run_experiment3.py',
    '--seed', '1',
    '--methods', 'FixedFedAvg', 'FixedTrimmedMean',
    '--num-attackers', '4',
    '--rounds', '20',
    '--resume',
    '--db', db_f4,
    '--data-root', '/content/fl_project/data'
], check=True)
backup_f4()


RUNNING BASELINES (f=4 Attackers): FixedFedAvg, FixedTrimmedMean


In [ ]:
print("=" * 70)
print("RUNNING AGENTIC AI (f=4 Attackers): Adaptive Pruning & Aggregation")
print("=" * 70)
subprocess.run([
    sys.executable, 'run_experiment3.py',
    '--seed', '1',
    '--methods', 'AgenticAI',
    '--num-attackers', '4',
    '--rounds', '20',
    '--resume',
    '--db', db_f4,
    '--data-root', '/content/fl_project/data'
], check=True)
backup_f4()
print("\n[PHASE 2 COMPLETE] f=4 runs finished!")


---
## Phase 3 — Results, Comparison Table & Breakdown Point Plot

In [ ]:
import sqlite3
import numpy as np
import matplotlib.pyplot as plt

methods = ['FixedFedAvg', 'FixedTrimmedMean', 'AgenticAI']
display_names = {
    'FixedFedAvg': 'Fixed FedAvg',
    'FixedTrimmedMean': 'Fixed Trimmed Mean (q=0.2)',
    'AgenticAI': 'AgenticAI (Ours)'
}

# 1. Fetch trajectories and Round 20 accuracies
def get_metrics_from_db(db_file):
    if not os.path.exists(db_file):
        return {}
    con = sqlite3.connect(db_file)
    cur = con.cursor()
    data = {}
    for m in methods:
        rows = cur.execute("""
            SELECT r.round, r.test_accuracy
            FROM experiment3_rounds r
            JOIN experiment3_runs runs ON r.run_id = runs.run_id
            WHERE runs.method = ? AND runs.seed = 1
            ORDER BY r.round ASC
        """, (m,)).fetchall()
        if rows:
            data[m] = {r: acc for r, acc in rows}
    con.close()
    return data

data_f3 = get_metrics_from_db('/content/fl_project/fl_metrics_byzantine_stress_f3.db')
data_f4 = get_metrics_from_db('/content/fl_project/fl_metrics_byzantine_stress_f4.db')

# Baseline f=2 results from Experiment 3 Seed 1:
# FixedFedAvg: 87.53%, FixedTrimmedMean: 90.64%, AgenticAI: 91.24%
f2_r20 = {'FixedFedAvg': 87.53, 'FixedTrimmedMean': 90.64, 'AgenticAI': 91.24}

print("=" * 80)
print("BREAKDOWN POINT COMPARISON TABLE (Seed 1, CIFAR-10, N=10)")
print("=" * 80)
print(f"{'Method':<30} | {'f=2 (20% Atk)':>14} | {'f=3 (30% Atk)':>14} | {'f=4 (40% Atk)':>14}")
print("-" * 80)
for m in methods:
    acc_f2 = f2_r20.get(m, 0.0)
    acc_f3 = data_f3.get(m, {}).get(20, None)
    acc_f4 = data_f4.get(m, {}).get(20, None)
    str_f3 = f"{acc_f3:>12.2f}%" if acc_f3 is not None else "     [N/A]  "
    str_f4 = f"{acc_f4:>12.2f}%" if acc_f4 is not None else "     [N/A]  "
    print(f"{display_names[m]:<30} | {acc_f2:>12.2f}% | {str_f3} | {str_f4}")
print("=" * 80)

# 2. Plot Breakdown Curves
f_levels = [2, 3, 4]
colors = {'FixedFedAvg': '#e74c3c', 'FixedTrimmedMean': '#3498db', 'AgenticAI': '#2ecc71'}
markers = {'FixedFedAvg': 'x', 'FixedTrimmedMean': 's', 'AgenticAI': 'o'}

plt.figure(figsize=(10, 5), dpi=120)
for m in methods:
    accs = [
        f2_r20.get(m, np.nan),
        data_f3.get(m, {}).get(20, np.nan),
        data_f4.get(m, {}).get(20, np.nan)
    ]
    plt.plot(f_levels, accs, label=display_names[m], color=colors[m], marker=markers[m], linewidth=2.5, markersize=8)

plt.title("Algorithmic Breakdown Point Comparison (N=10 Clients)", fontsize=14, fontweight='bold')
plt.xlabel("Number of Byzantine Attackers ($f$)", fontsize=12)
plt.ylabel("Final Test Accuracy at Round 20 (%)", fontsize=12)
plt.xticks([2, 3, 4], ["f=2 (20%)", "f=3 (30%)", "f=4 (40%)"])
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=2.0, color='gray', linestyle=':', label="TrimmedMean Breakdown Threshold (q=0.2)")
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("breakdown_point_comparison.png")
plt.show()


---
## 6. Download Stress Test Databases & Plots

In [ ]:
from google.colab import files

for db_name in ['fl_metrics_byzantine_stress_f3.db', 'fl_metrics_byzantine_stress_f4.db']:
    p = f"/content/fl_project/{db_name}"
    if os.path.exists(p):
        print(f"Triggering download for {db_name} ({os.path.getsize(p):,} bytes)...")
        files.download(p)

if os.path.exists("breakdown_point_comparison.png"):
    files.download("breakdown_point_comparison.png")
